# 프로젝트 2 - Weekend 1: RAG 평가 프레임워크와 하이브리드 검색

**프로젝트**: 검색형 RAG 기반 금융 상품(ETF) 추천 시스템

**학습 목표**:
1. ETF 데이터셋을 구축하고 벡터 스토어에 색인
2. Hit Rate, MRR, NDCG 등 검색 평가 지표 구현
3. BM25 + 벡터 하이브리드 검색으로 성능 개선
4. Query Expansion과 Multi-Query로 검색 품질 극대화

In [ ]:
# 환경 설정 및 라이브러리 설치
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu \
    rank_bm25 pandas numpy matplotlib gradio python-dotenv tiktoken

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

client = OpenAI()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("✅ 환경 설정 완료")

---
## 데이터 준비

아래 셀을 실행하면 ETF 금융 상품 데이터가 로드됩니다.

In [ ]:
# ============================================================
# ETF 금융 상품 샘플 데이터 (실습용)
# ============================================================
SAMPLE_ETF_DATA = [
    {"ticker": "KODEX200", "name": "KODEX 200", "category": "국내주식",
     "description": "KOSPI 200 지수를 추종하는 국내 대표 ETF. 삼성전자, SK하이닉스 등 대형주 중심으로 구성되어 있으며, 국내 주식시장 전체의 흐름을 반영합니다.",
     "expense_ratio": 0.15, "aum_billion": 58000, "risk_level": "중간",
     "returns": {"1m": 2.1, "3m": 5.4, "1y": 12.3, "3y": 28.5},
     "keywords": ["코스피", "대형주", "인덱스", "패시브"]},
    {"ticker": "TIGER미국S&P500", "name": "TIGER 미국 S&P500", "category": "해외주식",
     "description": "미국 S&P500 지수를 추종. 애플, 마이크로소프트, 아마존 등 미국 대형 기술주 포함. 환헤지 미적용으로 원/달러 환율 변동에 노출됩니다.",
     "expense_ratio": 0.07, "aum_billion": 42000, "risk_level": "중간",
     "returns": {"1m": 3.2, "3m": 8.1, "1y": 18.7, "3y": 45.2},
     "keywords": ["미국", "S&P500", "대형주", "기술주"]},
    {"ticker": "KODEX미국나스닥100", "name": "KODEX 미국나스닥100", "category": "해외주식",
     "description": "나스닥100 지수 추종. 기술 성장주 중심으로 애플, 엔비디아, 메타 등 포함. 고성장/고변동성 특성으로 공격적 투자자에게 적합합니다.",
     "expense_ratio": 0.09, "aum_billion": 35000, "risk_level": "높음",
     "returns": {"1m": 4.5, "3m": 12.3, "1y": 25.1, "3y": 62.8},
     "keywords": ["나스닥", "기술주", "성장주", "AI"]},
    {"ticker": "KODEX국고채10년", "name": "KODEX 국고채 10년", "category": "채권",
     "description": "대한민국 10년 만기 국고채를 추종하는 채권 ETF. 금리 인하 시 가격 상승, 안전자산 선호 시 수요 증가. 변동성이 낮아 안정적 투자에 적합합니다.",
     "expense_ratio": 0.05, "aum_billion": 12000, "risk_level": "낮음",
     "returns": {"1m": 0.8, "3m": 1.5, "1y": 4.2, "3y": 8.1},
     "keywords": ["국채", "채권", "안전자산", "금리"]},
    {"ticker": "TIGER금은혼합", "name": "TIGER 금은혼합", "category": "원자재",
     "description": "금과 은에 분산 투자하는 원자재 ETF. 인플레이션 헤지 수단으로 활용되며, 지정학적 리스크 시 안전자산으로 수요 증가합니다.",
     "expense_ratio": 0.39, "aum_billion": 3500, "risk_level": "중간",
     "returns": {"1m": 1.2, "3m": 3.8, "1y": 15.6, "3y": 35.2},
     "keywords": ["금", "은", "원자재", "인플레이션"]},
    {"ticker": "KODEX리츠", "name": "KODEX 한국부동산리츠인프라", "category": "부동산",
     "description": "국내 상장 리츠 및 인프라 펀드에 투자. 임대료 수입 기반의 안정적 배당 수익을 제공하며, 부동산 간접투자 수단으로 활용됩니다.",
     "expense_ratio": 0.09, "aum_billion": 4800, "risk_level": "중간",
     "returns": {"1m": -0.5, "3m": 2.1, "1y": 7.8, "3y": 15.3},
     "keywords": ["리츠", "부동산", "배당", "임대"]},
    {"ticker": "KODEX2차전지", "name": "KODEX 2차전지산업", "category": "테마",
     "description": "2차전지(배터리) 관련 기업에 집중 투자. LG에너지솔루션, 삼성SDI, 포스코퓨처엠 등 포함. 전기차 시장 성장에 따른 수혜가 기대됩니다.",
     "expense_ratio": 0.45, "aum_billion": 18000, "risk_level": "높음",
     "returns": {"1m": -2.3, "3m": -5.1, "1y": -12.4, "3y": 8.7},
     "keywords": ["2차전지", "배터리", "전기차", "테마"]},
    {"ticker": "TIGERBBD", "name": "TIGER 미국배당다우존스", "category": "배당",
     "description": "미국 고배당 우량주에 투자하는 ETF. 안정적인 배당 수익과 자본 이득을 동시에 추구합니다. 월배당 지급으로 현금흐름 관리에 유리합니다.",
     "expense_ratio": 0.01, "aum_billion": 52000, "risk_level": "낮음",
     "returns": {"1m": 1.8, "3m": 4.2, "1y": 10.5, "3y": 32.1},
     "keywords": ["배당", "미국", "월배당", "인컴"]},
    {"ticker": "KODEXKSM", "name": "KODEX 코스닥150", "category": "국내주식",
     "description": "코스닥 150 지수를 추종. 중소형 성장주 중심으로 바이오, IT, 게임 등 혁신 기업 포함. 코스피 대비 높은 변동성과 성장 잠재력을 가집니다.",
     "expense_ratio": 0.20, "aum_billion": 8500, "risk_level": "높음",
     "returns": {"1m": -1.2, "3m": 3.5, "1y": 8.9, "3y": 18.7},
     "keywords": ["코스닥", "중소형", "성장주", "바이오"]},
    {"ticker": "KOSEF단기자금", "name": "KOSEF 단기자금", "category": "머니마켓",
     "description": "초단기 채권 및 예금에 투자하는 MMF형 ETF. 원금 손실 위험이 극히 낮으며, 여유 자금 파킹 용도로 활용됩니다. 하루 단위 이자 발생.",
     "expense_ratio": 0.03, "aum_billion": 25000, "risk_level": "매우낮음",
     "returns": {"1m": 0.3, "3m": 0.9, "1y": 3.5, "3y": 10.2},
     "keywords": ["단기", "파킹", "안전", "예금"]},
]

# 사용자 프로필 데이터
SAMPLE_USER_PROFILES = [
    {"user_id": "U001", "name": "김초보", "level": "초보",
     "risk_tolerance": "낮음", "investment_goal": "안정적 수익",
     "monthly_budget": 500000, "preferred_categories": ["채권", "머니마켓"],
     "sample_queries": ["안전한 투자 상품 추천해주세요", "원금 손실 없는 ETF가 뭐가 있나요?"]},
    {"user_id": "U002", "name": "이중급", "level": "중급",
     "risk_tolerance": "중간", "investment_goal": "자산 증식",
     "monthly_budget": 2000000, "preferred_categories": ["국내주식", "해외주식"],
     "sample_queries": ["S&P500 추종 ETF 비교해주세요", "배당과 성장 균형 잡힌 포트폴리오 추천"]},
    {"user_id": "U003", "name": "박전문", "level": "전문",
     "risk_tolerance": "높음", "investment_goal": "공격적 수익",
     "monthly_budget": 10000000, "preferred_categories": ["테마", "해외주식"],
     "sample_queries": ["AI 관련 ETF 섹터 분석해줘", "나스닥100 vs 코스닥150 변동성 비교"]},
]

# 평가용 질의 데이터
SAMPLE_EVAL_QUERIES = [
    {"query": "초보자인데 안전한 투자 추천해주세요", "expected_tickers": ["KODEX국고채10년", "KOSEF단기자금", "TIGERBBD"]},
    {"query": "미국 기술주에 투자하고 싶어요", "expected_tickers": ["TIGER미국S&P500", "KODEX미국나스닥100"]},
    {"query": "월배당 받을 수 있는 ETF 있나요?", "expected_tickers": ["TIGERBBD", "KODEX리츠"]},
    {"query": "전기차 관련 투자 상품 알려주세요", "expected_tickers": ["KODEX2차전지"]},
    {"query": "인플레이션 헤지용 상품 추천", "expected_tickers": ["TIGER금은혼합", "KODEX리츠"]},
    {"query": "분산투자 포트폴리오 짜주세요", "expected_tickers": ["KODEX200", "TIGER미국S&P500", "KODEX국고채10년"]},
]

print(f"📦 ETF 데이터 로드 완료: {len(SAMPLE_ETF_DATA)}개 상품, {len(SAMPLE_USER_PROFILES)}개 프로필, {len(SAMPLE_EVAL_QUERIES)}개 평가 질의")
for cat in sorted(set(e["category"] for e in SAMPLE_ETF_DATA)):
    cnt = sum(1 for e in SAMPLE_ETF_DATA if e["category"] == cat)
    print(f"  - {cat}: {cnt}개")

---
## 문제 1: ETF 사용자 페르소나 정의

ETF 추천 시스템의 사용자 페르소나 3가지(초보/중급/전문)를 정의하세요.

**요구사항:**
- `personas` 딕셔너리: 키='초보'/'중급'/'전문'
- 각 페르소나에 `description`(설명)과 `queries`(질의 리스트) 포함
- 각 질의에 `query`(질문 텍스트)와 `expected_category`(기대 카테고리) 포함
- `project2_data/query_set.json`으로 저장

In [ ]:
# 문제 1: 사용자 페르소나 정의
personas = {
    "초보": {
        "description": "____",  # 투자 경험 설명
        "queries": [
            {"query": "____", "expected_category": ["____"]},
            # ... 3개 질의
        ]
    },
    # "중급": { ... },
    # "전문": { ... },
}

# JSON 저장
with open("project2_data/query_set.json", "w") as f:
    json.dump(personas, f, ensure_ascii=False, indent=2)

print(f"✅ {sum(len(p['queries']) for p in personas.values())}개 질의 저장 완료")

---
## 문제 2: 추가 ETF 문서 생성

5개 추가 ETF(ESG, 2차전지, 헬스케어, 리츠, 원자재)를 정의하고 LLM으로 설명을 생성하세요.

**요구사항:**
- `additional_etfs` 리스트: 5개 ETF (name, category, market)
- `risk_map` 딕셔너리: 카테고리별 위험도 매핑
- 각 ETF에 대해 `client.chat.completions.create()`로 설명 생성
- `etf_documents`에 추가 후 카테고리별 통계 출력

In [ ]:
# 문제 2: 추가 ETF 정의 및 문서 생성
additional_etfs = [
    {"name": "TIGER ESG리더스", "category": "ESG", "market": "국내"},
    # ... 4개 더 추가 (2차전지, 헬스케어, 리츠, 원자재)
]

risk_map = {
    # 카테고리: 위험도(1~5) 매핑
}

for etf in additional_etfs:
    # ---- 여기에 코드 작성 ----
    # 1) client.chat.completions.create()로 ETF 설명 생성
    # 2) etf_documents에 추가
    pass

df = pd.DataFrame(etf_documents)
print(f"카테고리별 수:\n{df['category'].value_counts()}")

---
## 문제 3: 멀티-관련성 질의 생성

하나의 질의에 여러 ETF가 관련되는 멀티-관련성 질의를 생성하세요.

**요구사항:**
- `multi_queries` 리스트: 각 항목에 `query`, `relevant`(관련 문서 리스트), `irrelevant` 포함
- `cats` 변수에 카테고리 분포 저장
- matplotlib로 카테고리 분포 바 차트 시각화
- CSV 파일로 저장

In [ ]:
# 문제 3: 멀티-관련성 질의 생성
multi_queries = [
    {
        "query": "____",
        "relevant": [{"doc_id": ____, "score": ____}],
        "irrelevant": [____]
    },
    # ... 최소 2개
]

# 카테고리 분포 시각화
cats = [item['category'] for item in eval_dataset]
# ---- 여기에 코드 작성 ----
# matplotlib 바 차트 + CSV 저장

---
## 문제 4: MMR 검색 구현

벡터 스토어에서 MMR(Maximal Marginal Relevance) 검색을 구현하고 유사도 검색과 비교하세요.

**요구사항:**
- `format_results(results, method_name)` 함수: 검색 결과를 포맷팅하여 출력
- Similarity Search와 MMR Search 각각 실행 후 시간 측정
- k값(3, 5, 10)별 카테고리 다양성 비교

In [ ]:
# 문제 4: MMR 검색 구현
import time

def format_results(results, method_name):
    # ---- 여기에 코드 작성 ----
    # 결과를 포맷팅하여 출력
    pass

query = "분산 투자에 적합한 ETF"

# Similarity Search
start = time.time()
sim_results = vectorstore.similarity_search_with_score(query, k=5)
# ---- 결과 출력 ----

# MMR Search
start = time.time()
mmr_results = vectorstore.max_marginal_relevance_search(query, k=5, fetch_k=10)
# ---- 결과 출력 + k값별 카테고리 비교 ----

---
## 문제 5: Precision@K와 Recall@K 구현

Precision@K와 Recall@K 평가 지표를 구현하세요.

**요구사항:**
- `precision_at_k(vectorstore, eval_data, k=5)` 함수 구현
- `recall_at_k(vectorstore, eval_data, k=5)` 함수 구현
- k=1~10에서 Hit Rate, MRR, Precision, Recall 4개 지표를 그래프로 비교

In [ ]:
# 문제 5: Precision@K, Recall@K 구현
def precision_at_k(vectorstore, eval_data, k=5):
    # ---- 여기에 코드 작성 ----
    # retrieved & relevant / k
    pass

def recall_at_k(vectorstore, eval_data, k=5):
    # ---- 여기에 코드 작성 ----
    # retrieved & relevant / len(relevant)
    pass

# k=1~10 그래프
ks = range(1, 11)
# ---- Hit Rate, MRR, Precision, Recall 4개 지표 비교 차트 ----

---
## 문제 6: NDCG 평가 리포트

Graded relevance(0, 1, 2, 3)를 사용한 NDCG 평가 리포트를 작성하세요.

**요구사항:**
- `report` 딕셔너리에 k=1,3,5,10별 Hit Rate, MRR, NDCG 저장
- JSON 파일로 저장 (`baseline_report.json`)
- 결과를 표 형태로 출력

In [ ]:
# 문제 6: NDCG 평가 리포트
report = {'baseline': {}}

for k in [1, 3, 5, 10]:
    # ---- 여기에 코드 작성 ----
    # hit_rate, mrr, ndcg 계산 후 report에 저장
    pass

# JSON 저장 + 결과 출력

---
## 문제 7: BM25 vs FAISS 비교 평가

BM25 검색 결과를 FAISS 검색 결과와 비교 평가하세요.

**요구사항:**
- `bm25_hit_rate(eval_data, k=5)` 함수 구현
- `bm25_mrr(eval_data, k=5)` 함수 구현
- BM25와 FAISS의 Hit Rate, MRR 비교 테이블 출력

In [ ]:
# 문제 7: BM25 vs FAISS 비교
def bm25_hit_rate(eval_data, k=5):
    # ---- 여기에 코드 작성 ----
    pass

def bm25_mrr(eval_data, k=5):
    # ---- 여기에 코드 작성 ----
    pass

# 비교 테이블 출력

---
## 문제 8: Alpha 최적화

Alpha 값을 0.0~1.0까지 0.1 단위로 변경하며 Hit Rate를 측정하고 최적값을 찾으세요.

**요구사항:**
- alpha를 0.0~1.0 범위에서 0.1 간격으로 탐색
- 각 alpha별 Hit Rate@5 출력
- 최적 alpha와 Hit Rate를 `checkpoint` 딕셔너리에 저장
- `project2_data/checkpoints/hybrid_config.json`으로 저장

In [ ]:
# 문제 8: Alpha 최적화
best_alpha, best_hr = 0, 0

for a in np.arange(0, 1.1, 0.1):
    # ---- 여기에 코드 작성 ----
    # hybrid_search()로 Hit Rate 측정
    pass

# 최적값 저장
checkpoint = {'best_alpha': best_alpha, 'best_hr': best_hr}
# JSON 저장

---
## 문제 9: 도메인 특화 동의어 사전

도메인 특화 동의어 사전을 만들고 쿼리 확장 성능을 비교하세요.

**요구사항:**
- `finance_synonyms` 딕셔너리: ETF, 배당, 안정, 성장, 미국 등의 동의어
- `synonym_expand(query)` 함수: 동의어로 확장된 쿼리 리스트 반환
- Baseline vs Synonym 확장 Hit Rate 비교

In [ ]:
# 문제 9: 도메인 동의어 사전
finance_synonyms = {
    'ETF': ['상장지수펀드', '____'],
    '배당': ['분배금', '____'],
    # ... 추가
}

def synonym_expand(query):
    # ---- 여기에 코드 작성 ----
    # query에 포함된 키워드를 동의어로 확장
    pass

# Baseline vs Synonym Hit Rate 비교

---
## 문제 10: 커스텀 Multi-Query Retriever

커스텀 프롬프트로 Multi-Query Retriever를 설정하세요.

**요구사항:**
- `custom_prompt` PromptTemplate 정의 (ETF 검색 전문가 역할)
- `retriever_custom` = MultiQueryRetriever 생성
- `retriever_custom.invoke()`로 검색 실행 후 결과 출력

In [ ]:
# 문제 10: 커스텀 Multi-Query Retriever
from langchain.prompts import PromptTemplate

custom_prompt = PromptTemplate(
    input_variables=['question'],
    template="""____"""  # ETF 검색 전문가 역할 프롬프트
)

# ---- 여기에 코드 작성 ----
# retriever_custom = MultiQueryRetriever.from_llm(...)
# results_custom = retriever_custom.invoke(...)

---
## 문제 11: 검색 비교 대시보드

모든 검색 방법의 결과를 나란히 비교하는 Gradio UI를 만드세요.

**요구사항:**
- `full_comparison(query, top_k)` 함수: FAISS/BM25/Hybrid 결과 비교
- `show_history()` 함수: 최근 검색 이력 표시
- `gr.Blocks`로 탭 UI 구성 (검색 탭 + 이력 탭)

In [ ]:
# 문제 11: 검색 비교 대시보드
import gradio as gr

search_history = []

def full_comparison(query, top_k):
    # ---- 여기에 코드 작성 ----
    # FAISS, BM25, Hybrid 3개 결과를 모두 비교
    pass

def show_history():
    # ---- 여기에 코드 작성 ----
    pass

# gr.Blocks로 탭 UI 구성

---

## Weekend 1 요약

### 달성한 것들
1. **ETF 데이터셋 구축**: 15개 ETF 문서 생성 및 벡터화
2. **평가 프레임워크**: Hit Rate, MRR, NDCG 구현
3. **BM25 검색**: 키워드 기반 검색 구축
4. **하이브리드 검색**: 벡터+BM25 앙상블
5. **고급 기법**: Query Expansion, Multi-Query
6. **Gradio 대시보드**: 인터랙티브 검색 UI

### Weekend 2 예고
- 리랭킹과 답변 품질 평가
- BLEU, ROUGE, BERTScore, LLM-as-Judge 평가
- 종합 평가 파이프라인과 품질 대시보드